In [32]:
#Import Libraries
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training


In [33]:
#Load Dataset
dataset = load_dataset("ag_news")

In [34]:
#Tokenization
tokenizer = DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-small")

#Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)



Map:   0%|          | 0/7600 [00:00<?, ? examples/s]Asking to pad to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no padding.
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
Map: 100%|██████████| 7600/7600 [00:01<00:00, 6301.19 examples/s]


In [35]:
#Load the Pretrained Model
model = DebertaV2ForSequenceClassification.from_pretrained("microsoft/deberta-v3-small", num_labels=4)



Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [38]:
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules = [
        "encoder.layer.*.attention.self.query",
        "encoder.layer.*.attention.self.key",
        "encoder.layer.*.attention.self.value",
        "encoder.layer.*.attention.output.dense",
        "encoder.layer.*.intermediate.dense",
        "encoder.layer.*.output.dense",
        "encoder.layer.*.attention.output.LayerNorm",
        "encoder.layer.*.output.LayerNorm",
        "embeddings.word_embeddings",
        "embeddings.position_embeddings"
    ]
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # To verify the trainable parameters

trainable params: 1,030,944 || all params: 142,928,932 || trainable%: 0.7213


In [42]:
#Train arguments
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy="epoch",
    learning_rate=2e-5,
    # per_device_train_batch_size=16,
    # per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
)

In [43]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

In [44]:
trainer.train()

TypeError: DebertaV2ForSequenceClassification.forward() got an unexpected keyword argument 'num_items_in_batch'

In [9]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import (
    get_peft_model, 
    LoraConfig, 
    TaskType,
    PeftModel
)
import evaluate
import numpy as np

# Load dataset
dataset = load_dataset("ag_news")

# Load tokenizer and model
model_name = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=4,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

# Add padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding=False, 
        truncation=True, 
        max_length=512
    )

# Tokenize dataset
tokenized_dataset = dataset.map(
    tokenize_function, 
    batched=True,
    remove_columns=["text"]
)

# Split dataset
train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]

# Data collator
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,
    max_length=512,
    return_tensors="pt"
)

# LoRA Configuration for "full" LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=16,  # Rank
    lora_alpha=32,  # LoRA alpha
    lora_dropout=0.1,  # LoRA dropout
    target_modules=[
        # DeBERTa v3 specific modules - we target all attention and feed-forward layers
        "query_proj",
        "key_proj", 
        "value_proj",
        "output_proj",
        "intermediate.dense",
        "output.dense",
        "encoder.layer.*.attention.self.*",
        "encoder.layer.*.attention.output.*",
        "encoder.layer.*.intermediate.*",
        "encoder.layer.*.output.*"
    ],
    bias="none",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# Load accuracy metric
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy_metric.compute(predictions=predictions, references=labels)

# Training arguments
training_args = TrainingArguments(
    output_dir="./deberta-v3-small-lora-agnews",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    # logging_dir="./logs",
    logging_steps=100,
    # report_to="None",  # Disable wandb/tensorboard if not needed
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=False,
)

# Create trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start training
print("Starting training...")
trainer.train()

# Save the model
trainer.save_model("./deberta-v3-small-lora-agnews-final")

# Evaluate the model
print("Evaluating model...")
results = trainer.evaluate()
print(f"Final evaluation results: {results}")

# Example inference
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    predicted_class = torch.argmax(predictions, dim=1).item()
    confidence = predictions[0][predicted_class].item()
    
    return predicted_class, confidence

# Test inference
test_text = "Apple announced new iPhone with advanced AI features"
pred_class, confidence = predict(test_text)
print(f"Predicted class: {pred_class}, Confidence: {confidence:.4f}")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-small and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 1,330,180 || all params: 143,228,168 || trainable%: 0.9287
Starting training...


c:\Users\matth\.conda\envs\FYP\Lib\site-packages\transformers\tokenization_utils_base.py:2774: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 